In [0]:
import os
import re
import pandas as pd

# ── Configuration ────────────────────────────────────────────────────────────
BASE_PATH = "/Volumes/[pe]_xx102_pe_finance/bronze/accrual_matching/input / data_proudcts_files"
CATALOG = "[pe]_xx102_pe_finance"
SCHEMA = "bronze"
TABLE_PREFIX = "raw_"  # Bronze layer naming convention
EXCLUDE_DIRS = {"companycode", "journalentryheader"}  # Directories to skip

# ── Set catalog & schema context ───────────────────────────────────────────
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

# ── Helper: sanitize column names for Delta compatibility ───────────────────
def sanitize_columns(df):
    """Replace spaces and special chars in column names with underscores."""
    for col_name in df.columns:
        clean_name = re.sub(r'[^a-zA-Z0-9_]', '_', col_name)
        clean_name = re.sub(r'_+', '_', clean_name).strip('_')
        if clean_name != col_name:
            df = df.withColumnRenamed(col_name, clean_name)
    return df

# ── Walk directories and process each CSV ───────────────────────────────────
# NOTE: Using pandas to read CSVs because Spark's path resolver treats
# square brackets in the Volume path as glob patterns.
results = []

for root, dirs, files in os.walk(BASE_PATH):
    # Skip excluded directories
    # dirs[:] = [d for d in dirs if d.lower() not in EXCLUDE_DIRS]
    csv_files = [f for f in files if f.lower().endswith('.csv')]
    for csv_file in csv_files:
        csv_path = os.path.join(root, csv_file)
        table_name = TABLE_PREFIX + os.path.splitext(csv_file)[0].lower()
        full_table_name = f"`{CATALOG}`.`{SCHEMA}`.{table_name}"

        try:
            # Read CSV via pandas (bypasses Spark glob on Volume path)
            pdf = pd.read_csv(csv_path, low_memory=False)

            # Convert to Spark DataFrame
            df = spark.createDataFrame(pdf)

            # Sanitize column names
            df = sanitize_columns(df)
            row_count = df.count()
            col_count = len(df.columns)

            # Write as Delta table (uses current catalog/schema context)
            df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)

            results.append((full_table_name, row_count, col_count, "SUCCESS"))
            print(f"\u2713 {full_table_name:60s} | {row_count:>8,} rows | {col_count:>4} cols")

        except Exception as e:
            results.append((full_table_name, 0, 0, f"FAILED: {str(e)[:100]}"))
            print(f"\u2717 {full_table_name:60s} | FAILED: {str(e)[:120]}")

# ── Summary ─────────────────────────────────────────────────────────────────
summary_df = pd.DataFrame(results, columns=["Table", "Rows", "Columns", "Status"])
success_count = (summary_df["Status"] == "SUCCESS").sum()
failed_count = len(summary_df) - success_count

print(f"\n{'═' * 70}")
print(f"  SUMMARY: {success_count} tables created successfully, {failed_count} failed")
print(f"  Total rows ingested: {summary_df.loc[summary_df['Status'] == 'SUCCESS', 'Rows'].sum():,}")
print(f"{'═' * 70}")

display(summary_df)

✓ `[pe]_xx102_pe_finance`.`bronze`.raw_companycode             |       49 rows |   26 cols
✓ `[pe]_xx102_pe_finance`.`bronze`.raw_companycodecurrencyrole |       98 rows |    7 cols
✓ `[pe]_xx102_pe_finance`.`bronze`.raw_companycodecurrencytranslation |        2 rows |   10 cols
✗ `[pe]_xx102_pe_finance`.`bronze`.raw_companycodehierarchy    | FAILED: [CANNOT_INFER_EMPTY_SCHEMA] Can not infer schema from empty dataset.
✗ `[pe]_xx102_pe_finance`.`bronze`.raw_companycodehierarchynode | FAILED: [CANNOT_INFER_EMPTY_SCHEMA] Can not infer schema from empty dataset.
✗ `[pe]_xx102_pe_finance`.`bronze`.raw_companycodehierarchynodetext | FAILED: [CANNOT_INFER_EMPTY_SCHEMA] Can not infer schema from empty dataset.
✗ `[pe]_xx102_pe_finance`.`bronze`.raw_companycodehierarchytext | FAILED: [CANNOT_INFER_EMPTY_SCHEMA] Can not infer schema from empty dataset.
✓ `[pe]_xx102_pe_finance`.`bronze`.raw_currencyrole            |       33 rows |    8 cols
✓ `[pe]_xx102_pe_finance`.`bronze`.raw_currencyroletex

Table,Rows,Columns,Status
`[pe]_xx102_pe_finance`.`bronze`.raw_companycode,49,26,SUCCESS
`[pe]_xx102_pe_finance`.`bronze`.raw_companycodecurrencyrole,98,7,SUCCESS
`[pe]_xx102_pe_finance`.`bronze`.raw_companycodecurrencytranslation,2,10,SUCCESS
`[pe]_xx102_pe_finance`.`bronze`.raw_companycodehierarchy,0,0,FAILED: [CANNOT_INFER_EMPTY_SCHEMA] Can not infer schema from empty dataset.
`[pe]_xx102_pe_finance`.`bronze`.raw_companycodehierarchynode,0,0,FAILED: [CANNOT_INFER_EMPTY_SCHEMA] Can not infer schema from empty dataset.
`[pe]_xx102_pe_finance`.`bronze`.raw_companycodehierarchynodetext,0,0,FAILED: [CANNOT_INFER_EMPTY_SCHEMA] Can not infer schema from empty dataset.
`[pe]_xx102_pe_finance`.`bronze`.raw_companycodehierarchytext,0,0,FAILED: [CANNOT_INFER_EMPTY_SCHEMA] Can not infer schema from empty dataset.
`[pe]_xx102_pe_finance`.`bronze`.raw_currencyrole,33,8,SUCCESS
`[pe]_xx102_pe_finance`.`bronze`.raw_currencyroletext,825,7,SUCCESS
`[pe]_xx102_pe_finance`.`bronze`.raw_costcenteractivitytype,30,27,SUCCESS
